# Görev 2: Gizli Katman, Çıkış Katmanı ve Manuel vs. F.cross_entropy Loss
**Kaynak:** Andrej Karpathy - *makemore Part 2: MLP* (18:35 - 37:56)

Gizli katmanı ve çıkış katmanını kur: embedding'leri düzleştir, W1 ve b1 ile tanh, W2 ve b2 ile logits. Loss'u geçen haftaki gibi elle hesapla, sonra F.cross_entropy ile aynı sonucu aldığını göster ve neden onu tercih ettiğimizi videodan anla.

In [1]:
import torch
import torch.nn.functional as F
import matplotlib.pyplot as plt
%matplotlib inline

words = open('names.txt', 'r').read().splitlines()
chars = sorted(list(set(''.join(words))))
stoi = {s:i+1 for i,s in enumerate(chars)}
stoi['.'] = 0
itos = {i:s for s,i in stoi.items()}

block_size = 3
X, Y = [], []
for w in words:
  context = [0] * block_size
  for ch in w + '.':
    ix = stoi[ch]
    X.append(context)
    Y.append(ix)
    context = context[1:] + [ix]
  
X = torch.tensor(X)
Y = torch.tensor(Y)

C = torch.randn((27, 2))
emb = C[X]
emb.shape

torch.Size([228146, 3, 2])

### Embedding'leri Düzleştirme: `torch.cat`, `unbind` ve `view`

In [ ]:
# Karpathy'nin gösterdiği yöntemler:
print(torch.cat([emb[:, 0, :], emb[:, 1, :], emb[:, 2, :]], 1).shape)
print(torch.cat(torch.unbind(emb, 1), 1).shape)
print(emb.view(-1, 6).shape)

# view() altta yatan Storage'ı kopyalamaz, sadece stride/shape bilgisini değiştirir!
torch.equal(torch.cat([emb[:, 0, :], emb[:, 1, :], emb[:, 2, :]], 1), emb.view(-1, 6))

### Gizli Katman (Hidden Layer)

In [ ]:
W1 = torch.randn((6, 100))
b1 = torch.randn(100)

In [ ]:
h = torch.tanh(emb.view(-1, 6) @ W1 + b1)
h.shape

### Çıkış Katmanı (Output Layer)

In [ ]:
W2 = torch.randn((100, 27))
b2 = torch.randn(27)

In [ ]:
logits = h @ W2 + b2
logits.shape

### Kaybı Geçen Haftaki Gibi Elle Hesaplama (Softmax + NLL)

In [ ]:
# 32 örneklik dilim üzerinde elle softmax ve negative log likelihood:
counts = logits[:32].exp()
prob = counts / counts.sum(1, keepdims=True)
loss_manual = -prob[torch.arange(32), Y[:32]].log().mean()
loss_manual

### `F.cross_entropy` ile Doğrulama

In [ ]:
loss_builtin = F.cross_entropy(logits[:32], Y[:32])
print("loss_manual: ", loss_manual.item())
print("loss_builtin:", loss_builtin.item())
print("İkisi tamamen eşit mi?:", torch.allclose(loss_manual, loss_builtin))

### Neden `F.cross_entropy` Tercih Ediyoruz? (Videodan)
1. **Geri Yayılım (Backpropagation) Verimliliği:** `counts`, `prob`, `log` gibi devasa tensörler bellekte tutulmaz. Geri yayılım tek bir C/CUDA çekirdeğinde analitik olarak türevi `prob - 1` formülüyle çok hızlı hesaplar.
2. **Sayısal Kararlılık (Numerical Stability):** Büyük logit değerlerinde `exp()` taşma (overflow $
ightarrow$ inf/nan) yapar. `F.cross_entropy` arka planda logit'lerden $\max$ değerini çıkarır: $\exp(x - \max(x))$ ve taşmayı kesinlikle önler.

In [ ]:
# Taşma (Overflow) Deneyi:
extreme_logits = torch.tensor([[10.0, 20.0, 1000.0]])
target = torch.tensor([2])

# Elle hesaplarsak: exp(1000) sonsuz (inf) olur
print("Elle exp:", extreme_logits.exp())
print("Elle prob:", extreme_logits.exp() / extreme_logits.exp().sum()) # nan!

# F.cross_entropy ile:
print("F.cross_entropy:", F.cross_entropy(extreme_logits, target).item()) # Taşma yok!